# 第26章：生产集成 — Autograd、torch.compile 与部署

## 前置知识
- 完成 Part 1-4

## 学习目标
- 将 Triton kernel 集成到 PyTorch 自动求导中
- 学会用 `torch.autograd.Function` 包装前向/反向
- 理解 `torch.compile` 与 Triton 的关系
- 掌握 Triton kernel 的生产部署模式

In [ ]:
import torch
import triton
import triton.language as tl
from torch.autograd import Function

## 26.1 torch.autograd.Function + Triton

要在训练中使用 Triton kernel，需要定义前向和反向传播：

```
前向: output = triton_forward(input)
                    ↓
            保存用于反向的张量 (ctx.save_for_backward)
                    ↓
反向: grad_input = triton_backward(grad_output)
```

In [ ]:
# 前向 kernel: ReLU
@triton.jit
def relu_forward_kernel(x_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, tl.maximum(x, 0.0), mask=mask)

# 反向 kernel: ReLU 的梯度是 1 if x > 0 else 0
@triton.jit
def relu_backward_kernel(x_ptr, grad_out_ptr, grad_in_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)
    grad_out = tl.load(grad_out_ptr + offs, mask=mask)
    # ReLU 梯度: grad_in = grad_out * (x > 0)
    grad_in = tl.where(x > 0, grad_out, 0.0)
    tl.store(grad_in_ptr + offs, grad_in, mask=mask)

In [ ]:
class TritonReLU(Function):
    """
    将 Triton kernel 包装为 PyTorch autograd Function。
    这样就能在训练中使用，自动求导系统会调用 backward。
    """
    @staticmethod
    def forward(ctx, x):
        # 保存输入，反向传播时需要
        ctx.save_for_backward(x)
        
        output = torch.empty_like(x)
        n = x.numel()
        BLOCK = 1024
        grid = (triton.cdiv(n, BLOCK),)
        relu_forward_kernel[grid](x, output, n, BLOCK=BLOCK)
        return output
    
    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        
        grad_input = torch.empty_like(x)
        n = x.numel()
        BLOCK = 1024
        grid = (triton.cdiv(n, BLOCK),)
        relu_backward_kernel[grid](x, grad_output, grad_input, n, BLOCK=BLOCK)
        return grad_input

# 使用方式
triton_relu = TritonReLU.apply

# 测试前向
x = torch.randn(1000, device='cuda', requires_grad=True)
y = triton_relu(x)
print(f"前向正确性: {torch.allclose(y, torch.relu(x))}")

# 测试反向
loss = y.sum()
loss.backward()
x_ref = x.detach().clone().requires_grad_(True)
torch.relu(x_ref).sum().backward()
print(f"反向正确性: {torch.allclose(x.grad, x_ref.grad)}")

## 26.2 更复杂的例子: Fused LayerNorm 带梯度

In [ ]:
@triton.jit
def layernorm_fwd_kernel(
    x_ptr, gamma_ptr, beta_ptr, out_ptr, mean_ptr, rstd_ptr,
    N, stride, eps,
    BLOCK: tl.constexpr,
):
    """LayerNorm 前向: 同时保存 mean 和 rstd 给反向用"""
    row = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    mask = offs < N
    
    x = tl.load(x_ptr + row * stride + offs, mask=mask, other=0.0).to(tl.float32)
    mean = tl.sum(x, axis=0) / N
    diff = tl.where(mask, x - mean, 0.0)
    var = tl.sum(diff * diff, axis=0) / N
    rstd = 1.0 / tl.sqrt(var + eps)
    
    gamma = tl.load(gamma_ptr + offs, mask=mask, other=1.0).to(tl.float32)
    beta = tl.load(beta_ptr + offs, mask=mask, other=0.0).to(tl.float32)
    out = gamma * diff * rstd + beta
    
    tl.store(out_ptr + row * stride + offs, out, mask=mask)
    # 保存统计量给反向
    tl.store(mean_ptr + row, mean)
    tl.store(rstd_ptr + row, rstd)

@triton.jit
def layernorm_bwd_kernel(
    grad_out_ptr, x_ptr, gamma_ptr, mean_ptr, rstd_ptr,
    grad_x_ptr, grad_gamma_ptr, grad_beta_ptr,
    N, stride,
    BLOCK: tl.constexpr,
):
    """LayerNorm 反向传播"""
    row = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    mask = offs < N
    
    grad_out = tl.load(grad_out_ptr + row * stride + offs, mask=mask, other=0.0).to(tl.float32)
    x = tl.load(x_ptr + row * stride + offs, mask=mask, other=0.0).to(tl.float32)
    gamma = tl.load(gamma_ptr + offs, mask=mask, other=1.0).to(tl.float32)
    mean = tl.load(mean_ptr + row)
    rstd = tl.load(rstd_ptr + row)
    
    xhat = (x - mean) * rstd
    
    # grad_gamma 和 grad_beta (需要跨行归约，这里用 atomic)
    wgrad = grad_out * xhat
    bgrad = grad_out
    tl.atomic_add(grad_gamma_ptr + offs, wgrad, mask=mask)
    tl.atomic_add(grad_beta_ptr + offs, bgrad, mask=mask)
    
    # grad_x
    dxhat = grad_out * gamma
    c1 = tl.sum(dxhat, axis=0) / N
    c2 = tl.sum(dxhat * xhat, axis=0) / N
    grad_x = (dxhat - c1 - xhat * c2) * rstd
    tl.store(grad_x_ptr + row * stride + offs, grad_x, mask=mask)

In [ ]:
class TritonLayerNorm(Function):
    @staticmethod
    def forward(ctx, x, gamma, beta, eps=1e-5):
        M, N = x.shape
        out = torch.empty_like(x)
        mean = torch.empty(M, device=x.device, dtype=torch.float32)
        rstd = torch.empty(M, device=x.device, dtype=torch.float32)
        
        BLOCK = triton.next_power_of_2(N)
        layernorm_fwd_kernel[(M,)](
            x, gamma, beta, out, mean, rstd,
            N, x.stride(0), eps, BLOCK=BLOCK,
        )
        ctx.save_for_backward(x, gamma, mean, rstd)
        ctx.N = N
        ctx.BLOCK = BLOCK
        return out
    
    @staticmethod
    def backward(ctx, grad_out):
        x, gamma, mean, rstd = ctx.saved_tensors
        M, N = x.shape
        
        grad_x = torch.empty_like(x)
        grad_gamma = torch.zeros_like(gamma, dtype=torch.float32)
        grad_beta = torch.zeros_like(gamma, dtype=torch.float32)
        
        layernorm_bwd_kernel[(M,)](
            grad_out, x, gamma, mean, rstd,
            grad_x, grad_gamma, grad_beta,
            N, x.stride(0), BLOCK=ctx.BLOCK,
        )
        return grad_x, grad_gamma.to(gamma.dtype), grad_beta.to(gamma.dtype), None

# 测试
M, N = 64, 256
x = torch.randn(M, N, device='cuda', requires_grad=True)
gamma = torch.ones(N, device='cuda', requires_grad=True)
beta = torch.zeros(N, device='cuda', requires_grad=True)

out = TritonLayerNorm.apply(x, gamma, beta)
out.sum().backward()

# 参考
x_ref = x.detach().clone().requires_grad_(True)
g_ref = gamma.detach().clone().requires_grad_(True)
b_ref = beta.detach().clone().requires_grad_(True)
torch.nn.functional.layer_norm(x_ref, (N,), g_ref, b_ref).sum().backward()

print(f"前向正确: {torch.allclose(out, torch.nn.functional.layer_norm(x.detach(), (N,), gamma.detach(), beta.detach()), atol=1e-4)}")
print(f"grad_x 正确: {torch.allclose(x.grad, x_ref.grad, atol=1e-3)}")

## 26.3 torch.compile 与 Triton

`torch.compile` 是 PyTorch 2.0 的核心特性，它会**自动生成 Triton kernel**！

```python
@torch.compile
def my_function(x):
    return torch.relu(x) + 1.0

# PyTorch 会自动:
# 1. 分析计算图
# 2. 融合可以融合的操作
# 3. 生成 Triton kernel
# 4. 编译并缓存
```

### torch.compile 的后端

```
torch.compile(fn)
    │
    ├── TorchDynamo (图捕获)
    │     捕获 Python 代码的计算图
    │
    ├── AOTAutograd (自动微分)
    │     前向/反向图分离
    │
    ├── TorchInductor (代码生成)
    │     ├── Triton kernels (GPU)
    │     └── C++ kernels (CPU)
    │
    └── 编译 & 缓存
```

In [ ]:
# torch.compile 自动生成 Triton kernel 的例子
def fused_ops_eager(x, weight, bias):
    """这些操作在 eager 模式下是 3 个独立 kernel"""
    y = x @ weight.T
    y = y + bias
    y = torch.relu(y)
    return y

# torch.compile 会自动融合这些操作
fused_ops_compiled = torch.compile(fused_ops_eager)

# 测试
M, N, K = 512, 256, 128
x = torch.randn(M, K, device='cuda')
weight = torch.randn(N, K, device='cuda')
bias = torch.randn(N, device='cuda')

# 第一次调用触发编译
out_compiled = fused_ops_compiled(x, weight, bias)
out_eager = fused_ops_eager(x, weight, bias)

print(f"torch.compile 正确性: {torch.allclose(out_compiled, out_eager, atol=1e-5)}")

# 性能对比
ms_eager = triton.testing.do_bench(lambda: fused_ops_eager(x, weight, bias))
ms_compiled = triton.testing.do_bench(lambda: fused_ops_compiled(x, weight, bias))
print(f"Eager: {ms_eager:.3f} ms")
print(f"Compiled: {ms_compiled:.3f} ms")
print(f"加速比: {ms_eager/ms_compiled:.2f}x")

## 26.4 自定义 Triton kernel + torch.compile

你可以注册自定义 Triton kernel 让 `torch.compile` 使用：

```python
# PyTorch 2.4+ 支持 triton.ops 注册
from torch.library import triton_op, wrap_triton

@triton_op("mylib::relu", mutates_args=())
def triton_relu(x: torch.Tensor) -> torch.Tensor:
    out = torch.empty_like(x)
    n = x.numel()
    grid = (triton.cdiv(n, 1024),)
    # wrap_triton 让 torch.compile 能看到这个 kernel
    wrap_triton(relu_forward_kernel)[grid](x, out, n, BLOCK=1024)
    return out
```

## 26.5 部署最佳实践

### 1. Kernel 预热
```python
# 第一次调用会触发编译，可能需要几秒
# 在生产环境中预热:
dummy = torch.randn(1, 1, device='cuda')
my_kernel[(1,)](dummy, dummy, 1, BLOCK=64)  # 预热
```

### 2. 缓存管理
```python
# Triton 编译结果缓存在 ~/.triton/cache/
# 生产环境中预编译并部署缓存目录
os.environ['TRITON_CACHE_DIR'] = '/app/triton_cache'
```

### 3. 打包为 Python 包
```python
# my_triton_ops/
# ├── __init__.py
# ├── kernels/
# │   ├── matmul.py     # Triton kernel 定义
# │   ├── layernorm.py
# │   └── attention.py
# └── ops.py             # 高层 API
```

### 4. 错误处理
```python
def safe_triton_matmul(a, b):
    try:
        return triton_matmul(a, b)
    except Exception:
        # 回退到 PyTorch
        return torch.matmul(a, b)
```

## 教程总结

恭喜你完成了全部 26 章！回顾学习路径：

```
Part 1: 基础           Part 2: 核心模式        Part 3: GEMM 优化
├── Hello Triton       ├── Softmax             ├── 共享内存
├── Program ID         ├── 朴素 GEMM           ├── Swizzle
├── 内存操作           ├── 分块 GEMM           ├── 流水线
├── Block 操作         ├── 算子融合            ├── Split-K
├── 控制流             └── 自动调优            ├── 混合精度
└── 数学运算                                   ├── Tensor Core
                                               └── 终极 GEMM

Part 4: FlashAttention  Part 5: 高级
├── 标准 Attention      ├── 调试分析
├── FA v1              ├── 编译器 IR
├── FA v2              └── 生产集成
├── FA v3
└── FA v4
```

### 你现在掌握的技能
1. 编写高性能 GPU kernel（不需要 CUDA）
2. 实现和优化矩阵乘法（达到 cuBLAS 90%+ 性能）
3. 实现 FlashAttention 全系列
4. 调试和分析 GPU kernel
5. 将 Triton kernel 集成到训练流水线

### 进一步学习
- [Triton 官方文档](https://triton-lang.org/)
- [Triton GitHub](https://github.com/triton-lang/triton)
- [FlashAttention 论文](https://arxiv.org/abs/2205.14135)
- [CUDA HGEMM 优化项目](本项目的 CUDA 版本)